# Étape 6 — Transformer de type BERT : prédire le mot masqué

```
Input  : "Rabat est [MASK] du Maroc"
              │
              ▼
   ┌────────────────────────────────────┐
   │ Bidirectional Encoder              │
   │ Representations from Transformers  │   encodeur uniquement, attention bidirectionnelle
   └────────────────────────────────────┘
              │
              ▼
Output : "capitale"
```

BERT utilise le contexte **à gauche et à droite** du masque. Il ne génère pas la suite d'un texte :
il complète une position masquée.

> Le premier lancement télécharge `bert-base-multilingual-cased` (~700 Mo), qui gère le français.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

## 1. Charger le tokenizer et le modèle préentraîné

In [2]:
nom_modele = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(nom_modele)
modele = AutoModelForMaskedLM.from_pretrained(nom_modele)
modele.eval()

print("Token de masque :", tokenizer.mask_token)
print("Paramètres :", f"{sum(p.numel() for p in modele.parameters()) / 1e6:.0f} M")
print("Couches :", modele.config.num_hidden_layers, "| têtes :", modele.config.num_attention_heads)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

C:\Users\yassi\Downloads\tp-architecture-transformer\tp-architecture-transformer\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yassi\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

C:\Users\yassi\Downloads\tp-architecture-transformer\tp-architecture-transformer\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yassi\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Token de masque : [MASK]
Paramètres : 178 M
Couches : 12 | têtes : 12


## 2. Préparer la phrase masquée

In [3]:
texte = f"Rabat est la {tokenizer.mask_token} du Maroc."
entrees = tokenizer(texte, return_tensors="pt")

print("Texte  :", texte)
print("Tokens :", tokenizer.convert_ids_to_tokens(entrees["input_ids"][0]))
print("Dimensions de l'entrée :", entrees["input_ids"].shape)

position_masque = (entrees["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()
print("Position du masque :", position_masque)

Texte  : Rabat est la [MASK] du Maroc.
Tokens : ['[CLS]', 'Rabat', 'est', 'la', '[MASK]', 'du', 'Maroc', '.', '[SEP]']
Dimensions de l'entrée : torch.Size([1, 9])
Position du masque : 4


## 3. Prédire le token masqué

In [4]:
with torch.no_grad():
    sortie = modele(**entrees)

print("Logits :", tuple(sortie.logits.shape))   # (1, L, taille du vocabulaire)
probabilites = torch.softmax(sortie.logits[0, position_masque], dim=-1)
top_probabilites, top_indices = torch.topk(probabilites, k=5)

print("\nPropositions de BERT :")
for p, i in zip(top_probabilites, top_indices):
    token = tokenizer.decode([i.item()]).strip()
    print(f"{p.item():6.2%} | {texte.replace(tokenizer.mask_token, token)}")

Logits : (1, 9, 119547)

Propositions de BERT :
86.06% | Rabat est la capitale du Maroc.
 5.19% | Rabat est la ville du Maroc.
 5.03% | Rabat est la préfecture du Maroc.
 0.85% | Rabat est la Rabat du Maroc.
 0.58% | Rabat est la capital du Maroc.


## 4. Une fonction réutilisable et d'autres exemples

In [5]:
def predire_masque(phrase, k=3):
    """phrase contient le mot-clé [MASK] exactement une fois."""
    phrase = phrase.replace("[MASK]", tokenizer.mask_token)
    entrees = tokenizer(phrase, return_tensors="pt")
    pos = (entrees["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0].item()
    with torch.no_grad():
        probas = torch.softmax(modele(**entrees).logits[0, pos], dim=-1)
    top = torch.topk(probas, k=k)
    return [(tokenizer.decode([i.item()]).strip(), p.item()) for p, i in zip(top.values, top.indices)]


exemples = [
    "Rabat est la [MASK] du Maroc.",
    "Casablanca est la plus grande [MASK] du Maroc.",
    "Paris est la capitale de la [MASK].",
    "Le soleil se lève à l'[MASK].",
    "The capital of Morocco is [MASK].",
]
for e in exemples:
    props = ", ".join(f"{t} ({p:.0%})" for t, p in predire_masque(e))
    print(f"{e:<50} -> {props}")

Rabat est la [MASK] du Maroc.                      -> capitale (86%), ville (5%), préfecture (5%)
Casablanca est la plus grande [MASK] du Maroc.     -> ville (62%), Casablanca (29%), port (2%)


Paris est la capitale de la [MASK].                -> France (41%), capitale (13%), ville (9%)


Le soleil se lève à l'[MASK].                      -> air (16%), intérieur (15%), est (12%)
The capital of Morocco is [MASK].                  -> Casablanca (45%), Rabat (32%), Morocco (7%)


## 5. Le contexte de droite compte

Même début de phrase, fin différente : BERT change sa prédiction, ce qu'un modèle de gauche à droite comme GPT ne pourrait pas faire.

In [6]:
for e in ["Le [MASK] aboie dans le jardin.", "Le [MASK] miaule dans le jardin."]:
    props = ", ".join(f"{t} ({p:.0%})" for t, p in predire_masque(e))
    print(f"{e:<40} -> {props}")

Le [MASK] aboie dans le jardin.          -> vent (4%), train (2%), sang (2%)
Le [MASK] miaule dans le jardin.         -> fruit (4%), vent (3%), feu (3%)


## Conclusion

- BERT est un **Transformer encodeur** : chaque position voit toute la phrase (attention bidirectionnelle).
- Il est préentraîné en **modélisation du langage masqué** : retrouver des tokens cachés.
- Il est adapté à la compréhension (classification, extraction, complétion de masque), pas à la génération libre comme GPT.

➡️ Deuxième partie : `exercices/`